## LightGBM

Main model. Same data preparation and train/test split as `02_baseline`, so the test set is identical.
Tree models handle missing values and outliers natively: no imputation, clipping or scaling.
Model selection uses 5-fold stratified cross-validation on train only. The test set is used once, for the final comparison.

**1 - Install LightGBM**

In [0]:
%pip install lightgbm
dbutils.library.restartPython()

**2 - Imports and constants**

In [0]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score

PATH = "/Volumes/workspace/risco_credito/raw/cs-training.csv"
TARGET = "SeriousDlqin2yrs"
PAST_DUE = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]
RANDOM_STATE = 42
N_SPLITS = 5

# Test-set results from 02_baseline (logistic regression)
BASELINE = {"roc_auc": 0.8601, "pr_auc": 0.3977}

**3 - Read CSV, mark missing values and split**

In [0]:
df = pd.read_csv(PATH, index_col=0)

# Flags first (row-level rules, no statistics -> safe before split)
df["past_due_sentinel"] = df[PAST_DUE].isin([96, 98]).any(axis=1).astype(int)
df["income_missing"] = df["MonthlyIncome"].isna().astype(int)
df["dependents_missing"] = df["NumberOfDependents"].isna().astype(int)

# Invalid values -> NaN (kept as NaN: LightGBM handles missing values natively)
df.loc[df["age"] == 0, "age"] = np.nan
df[PAST_DUE] = df[PAST_DUE].where(~df[PAST_DUE].isin([96, 98]))

X = df.drop(columns=TARGET)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"train: {X_train.shape} | default rate {y_train.mean():.4f}")
print(f"test:  {X_test.shape} | default rate {y_test.mean():.4f}")

**4 - Cross-validate LightGBM on train**

In [0]:
from sklearn.model_selection import cross_validate

lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,          # default tree size
    random_state=RANDOM_STATE,
    verbose=-1,             # silence training logs
)

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

scores = cross_validate(
    lgbm, X_train, y_train, cv=cv,
    scoring={"roc_auc": "roc_auc", "pr_auc": "average_precision"},
)

for m in ["roc_auc", "pr_auc"]:
    s = scores[f"test_{m}"]  # "test_" = validation fold, not our held-out test set
    print(f"{m}: {s.mean():.4f} ± {s.std():.4f} | folds {np.round(s, 4)} | baseline (test) {BASELINE[m]:.4f}")

**5 - Find the number of trees with early stopping**

In [0]:
import lightgbm as lgb

best_iters, roc_scores, pr_scores = [], [], []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_train, y_train), start=1):
    X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
    y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]

    model = LGBMClassifier(
        n_estimators=2000,            # upper bound; early stopping picks the real number
        learning_rate=0.05,
        num_leaves=31,
        metric="average_precision",   # stop on PR-AUC, our main metric
        random_state=RANDOM_STATE,
        verbose=-1,
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)],
    )

    proba = model.predict_proba(X_va)[:, 1]  # uses best_iteration_ automatically
    best_iters.append(model.best_iteration_)
    roc_scores.append(roc_auc_score(y_va, proba))
    pr_scores.append(average_precision_score(y_va, proba))
    print(f"fold {fold}: best_iter {model.best_iteration_:4d} | roc_auc {roc_scores[-1]:.4f} | pr_auc {pr_scores[-1]:.4f}")

print(f"\nroc_auc: {np.mean(roc_scores):.4f} ± {np.std(roc_scores):.4f} | baseline (test) {BASELINE['roc_auc']:.4f}")
print(f"pr_auc:  {np.mean(pr_scores):.4f} ± {np.std(pr_scores):.4f} | baseline (test) {BASELINE['pr_auc']:.4f}")
print(f"best_iteration: mean {np.mean(best_iters):.0f} | median {np.median(best_iters):.0f}")

**6 - Train final LightGBM and evaluate once on test**

In [0]:
N_TREES = int(np.median(best_iters))

lgbm_final = LGBMClassifier(
    n_estimators=N_TREES,     # chosen by early stopping in CV (median of folds)
    learning_rate=0.05,
    num_leaves=31,
    random_state=RANDOM_STATE,
    verbose=-1,
)
lgbm_final.fit(X_train, y_train)

proba_test = lgbm_final.predict_proba(X_test)[:, 1]
lgbm_test = {
    "roc_auc": roc_auc_score(y_test, proba_test),
    "pr_auc": average_precision_score(y_test, proba_test),
}

results = pd.DataFrame({"logistic (02)": BASELINE, "lightgbm": lgbm_test})
results["lift"] = results["lightgbm"] - results["logistic (02)"]
print(f"n_estimators = {N_TREES}\n")
print(results.round(4))

**7 - Log the final model to MLflow**

In [0]:
import mlflow
from mlflow.models import infer_signature

signature = infer_signature(X_train, lgbm_final.predict_proba(X_train)[:, 1])

with mlflow.start_run(run_name="lightgbm_final") as run:
    mlflow.log_params({
        "model": "LGBMClassifier",
        "n_estimators": N_TREES,           # median best_iter from 5-fold CV
        "learning_rate": 0.05,
        "num_leaves": 31,
        "early_stopping_rounds": 100,
        "random_state": RANDOM_STATE,
    })
    mlflow.log_metrics({
        "test_roc_auc": lgbm_test["roc_auc"],
        "test_pr_auc": lgbm_test["pr_auc"],
        "cv_roc_auc": np.mean(roc_scores),
        "cv_pr_auc": np.mean(pr_scores),
    })
    mlflow.lightgbm.log_model(
        lgbm_final,
        name="model",
        signature=signature,
        input_example=X_train.head(5),
    )

print(f"run_id: {run.info.run_id}")